<a href="https://colab.research.google.com/github/yojith712/python/blob/main/ISEF_AI_Code.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
import pandas as pd

DEMO_REFERENCE = pd.DataFrame([
    {
        "gene": "CFTR",
        "variant_id": "DEMO-CFTR-001",
        "label": "illustrative disease-associated example",
        "evidence": "Fictional teaching record; not a real variant assertion.",
    },
    {
        "gene": "TP53",
        "variant_id": "DEMO-TP53-001",
        "label": "illustrative cancer-related example",
        "evidence": "Fictional teaching record; not a real variant assertion.",
    },
])

def _clean(value):
    if pd.isna(value):
        return ""
    return str(value).strip()

def _prepare_table(table, required_columns, table_name):
    if not isinstance(table, pd.DataFrame):
        raise TypeError(f"{table_name} must be a pandas DataFrame.")
    prepared = table.copy()
    prepared.columns = [str(column).replace("\ufeff", "").strip().lower() for column in prepared.columns]
    missing = sorted(set(required_columns) - set(prepared.columns))
    if missing:
        raise ValueError(f"{table_name} is missing required columns: {', '.join(missing)}")
    return prepared.loc[:, required_columns].copy()

def scan_variants(samples, reference):
    """Return exact gene/variant-ID matches; this does not assess clinical meaning."""
    samples = _prepare_table(samples, ["gene", "variant_id"], "Input table")
    reference = _prepare_table(
        reference, ["gene", "variant_id", "label", "evidence"], "Reference table"
    )

    lookup = {}
    for row in reference.itertuples(index=False):
        key = (_clean(row.gene).upper(), _clean(row.variant_id).upper())
        lookup[key] = {"reference_label": _clean(row.label), "evidence": _clean(row.evidence)}

    results = []
    for row in samples.itertuples(index=False):
        gene = _clean(row.gene)
        variant_id = _clean(row.variant_id)
        match = lookup.get((gene.upper(), variant_id.upper()))
        results.append({
            "gene": gene,
            "variant_id": variant_id,
            "result": "REFERENCE MATCH" if match else "NO EXACT MATCH",
            "reference_label": match["reference_label"] if match else "",
            "evidence": match["evidence"] if match else "Not present in this reference table.",
        })
    return pd.DataFrame(results, columns=[
        "gene", "variant_id", "result", "reference_label", "evidence"
    ])

print("Scanner ready. The built-in reference rows are fictional teaching examples.")


Scanner ready. The built-in reference rows are fictional teaching examples.


In [5]:
sample_variants = pd.DataFrame([
    {"gene": "CFTR", "variant_id": "DEMO-CFTR-001"},
    {"gene": "TP53", "variant_id": "DEMO-TP53-001"},
    {"gene": "CFTR", "variant_id": "DEMO-NOT-IN-REFERENCE"},
])
reference_variants = DEMO_REFERENCE.copy()
demo_results = scan_variants(sample_variants, reference_variants)
display(demo_results)


,gene,variant_id,result,reference_label,evidence
0,CFTR,DEMO-CFTR-001,REFERENCE MATCH,illustrative disease-associated example,Fictional teaching record; not a real variant ...
1,TP53,DEMO-TP53-001,REFERENCE MATCH,illustrative cancer-related example,Fictional teaching record; not a real variant ...
2,CFTR,DEMO-NOT-IN-REFERENCE,NO EXACT MATCH,,Not present in this reference table.


In [9]:
from google.colab import sheets
sheet = sheets.InteractiveSheet()

MessageError: Error: credential propagation was unsuccessful

In [6]:
from google.colab import sheets
sheet = sheets.InteractiveSheet(df=demo_results)

MessageError: Error: credential propagation was unsuccessful

In [ ]:
from google.colab import files
import io

uploaded_input = files.upload()
if not uploaded_input:
    raise ValueError("No input file was uploaded.")
input_filename = next(iter(uploaded_input))
sample_variants = pd.read_csv(io.BytesIO(uploaded_input[input_filename]))
print(f"Loaded input file: {input_filename}")
display(sample_variants.head())


In [ ]:
uploaded_reference = files.upload()
if not uploaded_reference:
    raise ValueError("No reference file was uploaded.")
reference_filename = next(iter(uploaded_reference))
reference_variants = pd.read_csv(io.BytesIO(uploaded_reference[reference_filename]))
print(f"Loaded reference file: {reference_filename}")
display(reference_variants.head())


In [ ]:
results = scan_variants(sample_variants, reference_variants)
display(results)

report_path = "variant_scan_report.csv"
results.to_csv(report_path, index=False)
from google.colab import files
files.download(report_path)


In [3]:
%pip -q install biopython

from Bio import Entrez, SeqIO
from IPython.display import display, Markdown

# Fetch the public reference transcript record from NCBI RefSeq.
handle = Entrez.efetch(db="nuccore", id="NM_000492.4", rettype="gb", retmode="text")
record = SeqIO.read(handle, "genbank")
handle.close()
cds_features = [feature for feature in record.features if feature.type == "CDS"]
if len(cds_features) != 1:
    raise RuntimeError("Expected one coding sequence in the CFTR reference record.")
CFTR_REFERENCE_CDS = str(cds_features[0].extract(record.seq)).upper()

# In NM_000492.4 coding-DNA numbering, c.1521_1523 is the sequence CTT.
if CFTR_REFERENCE_CDS[1520:1523] != "CTT":
    raise RuntimeError("The fetched reference does not match the expected CFTR reference at c.1521_1523.")

print(f"Loaded {record.id}; coding sequence length: {len(CFTR_REFERENCE_CDS)} bases")
print("Reference orientation: coding strand, 5′ to 3′. No individual sample data is used.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 32.5 MB/s eta 0:00:00


/usr/local/lib/python3.13/dist-packages/Bio/Entrez/__init__.py:693: UserWarning: 
            Email address is not specified.

            To make use of NCBI's E-utilities, NCBI requires you to specify your
            email address with each request.  As an example, if your email address
            is A.N.Other@example.com, you can specify it as follows:
               from Bio import Entrez
               Entrez.email = 'A.N.Other@example.com'
            In case of excessive usage of the E-utilities, NCBI will attempt to contact
            a user at the email address provided before blocking access to the
            E-utilities.
  warnings.warn(


Loaded NM_000492.4; coding sequence length: 4443 bases
Reference orientation: coding strand, 5′ to 3′. No individual sample data is used.


In [ ]:
# Built-in synthetic example: delete the three bases for CFTR c.1521_1523delCTT.
hypothetical_cds = CFTR_REFERENCE_CDS[:1520] + CFTR_REFERENCE_CDS[1523:]

# To try your own hypothetical sequence, replace the line above with:
# hypothetical_cds = "PASTE YOUR FULL SIMULATED CFTR CODING STRAND HERE"

def clean_dna(sequence, name):
    sequence = "".join(str(sequence).split()).upper()
    invalid = sorted(set(sequence) - set("ACGT"))
    if invalid:
        raise ValueError(f"{name} has non-ACGT characters: {invalid}")
    return sequence

def assess_cftr_hypothesis(reference, alternate):
    reference = clean_dna(reference, "Reference sequence")
    alternate = clean_dna(alternate, "Hypothetical sequence")
    if reference == alternate:
        return {
            "gene": "CFTR",
            "sequence_change": "No difference detected",
            "catalog_result": "No disease association returned",
            "explanation": "The hypothetical sequence matches the reference exactly.",
        }

    known_f508del = reference[:1520] + reference[1523:]
    if alternate == known_f508del:
        return {
            "gene": "CFTR",
            "sequence_change": "NM_000492.4:c.1521_1523delCTT (p.Phe508del; F508del)",
            "catalog_result": "Known association: cystic fibrosis",
            "explanation": (
                "ClinVar classifies this exact CFTR variant as pathogenic for cystic fibrosis. "
                "Cystic fibrosis is usually autosomal recessive; detecting one altered copy alone "
                "does not establish that the disease would occur."
            ),
        }

    return {
        "gene": "CFTR",
        "sequence_change": "Sequence differs from the reference",
        "catalog_result": "Not classified by this demo",
        "explanation": (
            "This notebook recognizes only the exact single F508del example. "
            "An unrecognized change is unknown here; it is not evidence that the change is benign or disease-causing."
        ),
    }

sequence_result = assess_cftr_hypothesis(CFTR_REFERENCE_CDS, hypothetical_cds)
display(pd.DataFrame([sequence_result]))